# Cusanovich et al. (2018) -- A Single-Cell Atlas of In Vivo Mammalian Chromatin Accessibility

Source: Cusanovich et al., "A Single-Cell Atlas of In Vivo Mammalian Chromatin Accessibility." Cell 174.5 (2018): 1309-1324. GEO accession GSE111586.

sci-ATAC-seq profiling of 81,173 nuclei across 13 adult mouse tissues, yielding 436,206 chromatin accessibility peaks after quality filtering. The source publication also reports 40 marker-based cell labels, one of which (Unknown) covers cells the source could not annotate and is dropped throughout this notebook.

The reference label used here is tissue of dissection, not the marker-based cell label. Tissue is recorded at the time of nucleus collection and is therefore independent of any clustering performed on the accessibility matrix, the same kind of external ground truth Klein's collection timepoints provide.

---

## 0. Imports & Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from benchmarks._studies import (
    STUDIES,
    carve_cache_path,
    cvi_sweep,
    fit_or_load_carve,
    load_study,
    study_model_grids,
)
from benchmarks.figures import figure_cusanovich_results, prepare_composite
from benchmarks.figures._paths import CASE_STUDY_DIR

RANDOM_SEED = 42
SCALE = "dev"  # switch to "publication" for the manuscript run

# Figures are written under a scale-qualified directory, so a development
# figure can never be mistaken for one that belongs in the manuscript. Only
# the publication directory is ever copied into overleaf/vis/.
OUT_DIR = CASE_STUDY_DIR if SCALE == "publication" else CASE_STUDY_DIR / SCALE
OUT_DIR.mkdir(parents=True, exist_ok=True)

study = STUDIES["cusanovich"]
model_grids = study_model_grids(study)
X, y, meta = load_study(study, scale=SCALE)
print(meta["n_cells"], "cells,", meta["n_features"], "components,", meta["scale"])


def show(fig):
    """Display a figure exactly once, then close it."""
    display(fig)
    plt.close(fig)

---

## 1. Baseline Metrics

We evaluate Silhouette, Gap statistic, Davies-Bouldin, and Calinski-Harabasz across k = 4, ..., 16 for KMeans and spectral clustering (self-tuning affinity):

In [ ]:
curves_df, best_df = cvi_sweep(
    X, y, model_grids=model_grids, candidate_k=study.candidate_k,
    random_state=RANDOM_SEED, n_jobs=-1,
)
best_df

---

## 2. CARVE Analysis

In [ ]:
carve = fit_or_load_carve(
    X, y,
    cache_path=carve_cache_path(
        study, scale=SCALE, root=Path("./carve_state_saves")
    ),
    model_grids=model_grids,
    random_state=RANDOM_SEED,
    consensus_anchors=study.consensus_anchors,
)

---

## 3. Quantitative Comparison

### 3.1 Composite Paper Figure (with ARI Comparison)

In [ ]:
inputs = prepare_composite(
    X, y.to_numpy(), carve, curves_df=curves_df, best_df=best_df,
    comparison_metric="silhouette", measure="stability", rule="1se",
    random_state=RANDOM_SEED,
)
show(figure_cusanovich_results(inputs, out_dir=OUT_DIR))

---

## 4. Full-Atlas Validation

Spectral clustering and Ward agglomerative clustering both require a dense pairwise matrix over all cells, which does not fit at atlas scale. This pass instead sweeps Leiden resolution over every annotated cell in the atlas and checks that the subsample's conclusion survives past the subsample. It runs only when SCALE is set to publication.

The atlas is published at 81,173 cells; cell_label=="Unknown" cells (roughly a third of the atlas) are dropped throughout this notebook, so the annotated set analyzed here is smaller than that published count. See atlas_meta["n_cells_annotated"] below for the number actually used.

In [ ]:
from carve import CARVE

from benchmarks._studies import study_resolution_grids

if SCALE == "publication":
    X_atlas, y_atlas, atlas_meta = load_study(study, scale="atlas")
    print(
        atlas_meta["n_cells"],
        "annotated cells at atlas scale, out of",
        atlas_meta["n_cells_full"],
        "in the published atlas",
    )

    atlas = CARVE(
        estimator_param_grids=study_resolution_grids(study),
        n_resamples=100,
        n_jobs=1,
        random_state=RANDOM_SEED,
        consensus_anchors=study.consensus_anchors,
    ).fit(X_atlas)
    display(
        atlas.estimator_results_[["config_id", "resolution", "ari_stability"]]
    )

---

## 5. Summary

Cusanovich et al. report three levels of granularity for the mouse sci-ATAC atlas: 13 dissected tissues, a working partition of 30 unsupervised clusters, and 40 marker-annotated cell labels. CARVE's stability-selected k is compared against all three; because the finest partition (40 cell labels) is not expected to be the most stable one a resampling procedure recovers, a selected k well below 40 is the expected finding here, not a limitation. Panel F of the composite figure above reports the agreement (ARI) between CARVE's clustering and the reported tissue labels directly, and the full-atlas pass shows whether that agreement holds across the full annotated set (atlas scale) rather than only in the case-study subsample. The atlas is published at 81,173 cells; the annotated set analyzed at atlas scale is smaller, since cell_label=="Unknown" cells are dropped throughout this notebook.

---